# ECHO — Investigation Notebook
**Entangled Minds: The Quantathon · Qubits Qrew, NED**

### Setup — run first

In [1]:
!pip install qiskit qiskit-aer pylatexenc matplotlib --quiet

from qiskit import QuantumCircuit, transpile, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import numpy as np, matplotlib.pyplot as plt

sim = AerSimulator()
def run(circuit, shots=2000):
    return sim.run(transpile(circuit, sim), shots=shots).result().get_counts()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 1.9 MB/s eta 0:00:00


### Module 01 — The Gatekeeper

In [2]:
# BRONZE 1: Always produce 1
qc1 = QuantumCircuit(1, 1)
qc1.x(0)
qc1.measure(0, 0)
print("Bronze 1:", run(qc1))

# BRONZE 2: Produce a 50/50 result
qc2 = QuantumCircuit(1, 1)
qc2.h(0)
qc2.measure(0, 0)
print("Bronze 2:", run(qc2))

# SILVER: H followed by H returns the qubit to 0
qc3 = QuantumCircuit(1, 1)
qc3.h(0)
qc3.h(0)
qc3.measure(0, 0)
print("Silver:", run(qc3))

# GOLD: Produce 101 from 000
qc4 = QuantumCircuit(3, 3)
qc4.x(0)
qc4.x(2)
qc4.measure([0, 1, 2], [0, 1, 2])
print("Gold:", run(qc4))

Bronze 1: {'1': 2000}
Bronze 2: {'0': 995, '1': 1005}
Silver: {'0': 2000}
Gold: {'101': 2000}


The results demonstrate how quantum gates can create deterministic bits, a 50/50 measurement, and the bit pattern `101`.

### Module 02 — The Honest Coin  *(stretch)*

In [ ]:
# the recovered seed circuit — run it as-is
seed = QuantumCircuit(1, 1)
seed.ry(1.159, 0)
seed.measure(0, 0)
print(run(seed))

### Module 03 — The Two Observers

In [10]:
# BRONZE: Create an entangled pair that produces 00 or 11
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print("Bronze:", run(qc))

# SILVER: Classical shared secret coin
import random

counts = {"00": 0, "11": 0}

for _ in range(2000):
    secret_bit = random.randint(0, 1)
    answer = str(secret_bit) * 2
    counts[answer] += 1

print("Silver:", counts)

# GOLD: CHSH Bell test
import math

def bell_test(a, b):
    qc = QuantumCircuit(2, 2)

    # Create an entangled pair
    qc.h(0)
    qc.cx(0, 1)

    # Apply measurement settings
    qc.ry(-a, 0)
    qc.ry(-b, 1)

    qc.measure([0, 1], [0, 1])
    counts = run(qc, shots=10000)

    total = sum(counts.values())

    same = sum(v for k, v in counts.items()
               if k.replace(" ", "") in ("00", "11"))

    different = total - same

    return (same - different) / total


# CHSH measurement angles
a0 = 0
a1 = math.pi / 2
b0 = math.pi / 4
b1 = -math.pi / 4

e00 = bell_test(a0, b0)
e01 = bell_test(a0, b1)
e10 = bell_test(a1, b0)
e11 = bell_test(a1, b1)

chsh = abs(e00 + e01 + e10 - e11)

print("CHSH score:", chsh)
print("Classical limit: 2")
print("Quantum maximum:", 2 * math.sqrt(2))

Bronze: {'00': 983, '11': 1017}
Silver: {'00': 1005, '11': 995}
CHSH score: 2.8194
Classical limit: 2
Quantum maximum: 2.8284271247461903


The correlated measurement results and CHSH score of 2.8194, which exceeds the classical limit of 2, demonstrate quantum correlations beyond the classical CHSH bound.

### Module 04 — The Erased Measurement  *(stretch)*

In [ ]:
qc = QuantumCircuit(1, 1)

qc.measure(0, 0)
print(run(qc))

### Module 05 — The Damaged Message

In [7]:
stream = "100011001001001011100110001011000000001100011011000111000000110000001000000111001010101011111111"

# BRONZE: Recover the original message using majority vote
triples = [stream[i:i+3] for i in range(0, len(stream), 3)]

message = ""
for triple in triples:
    if triple.count("1") >= 2:
        message += "1"
    else:
        message += "0"

print("Bronze - Recovered bits:", message)
print("Bronze - Message:", bytes.fromhex(
    format(int(message, 2), "032b")
    .encode().hex()
).decode("ascii") if False else "".join(
    chr(int(message[i:i+8], 2))
    for i in range(0, len(message), 8)
))

# SILVER: Show where majority voting fails
damaged_triple = "110"
print("Silver - Damaged triple:", damaged_triple)
print("Silver - Majority result:",
      "1" if damaged_triple.count("1") >= 2 else "0")
print("Silver - Correct original bit was 0")

# GOLD: Explain correction and detection limits
print("Gold - Corrects up to 1 flipped bit per triple")
print("Gold - Can detect 1 or 2 flips if errors are limited to 2")
print("Gold - 3 flips can change 000 into 111 or 111 into 000")
print("Gold - Unknown quantum states cannot be copied perfectly")

Bronze - Recovered bits: 01000101010000110100100001001111
Bronze - Message: ECHO
Silver - Damaged triple: 110
Silver - Majority result: 1
Silver - Correct original bit was 0
Gold - Corrects up to 1 flipped bit per triple
Gold - Can detect 1 or 2 flips if errors are limited to 2
Gold - 3 flips can change 000 into 111 or 111 into 000
Gold - Unknown quantum states cannot be copied perfectly


Majority voting recovers the original message `ECHO` when at most one bit per triple is flipped, while excessive errors can cause incorrect decoding.

### Module 06 — The Impossible Claim  *(stretch)*

In [9]:
a = 1.0

def teleport(with_correction=True):
    q = QuantumRegister(3)
    c0 = ClassicalRegister(1, 'c0')
    c1 = ClassicalRegister(1, 'c1')
    co = ClassicalRegister(1, 'co')

    qc = QuantumCircuit(q, c0, c1, co)

    # Prepare the message
    qc.ry(a, q[0])

    # Create an entangled pair
    qc.h(q[1])
    qc.cx(q[1], q[2])

    # Bell measurement
    qc.cx(q[0], q[1])
    qc.h(q[0])
    qc.measure(q[0], c0)
    qc.measure(q[1], c1)

    # Apply corrections using classical measurement results
    if with_correction:
        with qc.if_test((c1, 1)):
            qc.x(q[2])
        with qc.if_test((c0, 1)):
            qc.z(q[2])

    # Check whether the original state arrived
    qc.ry(-a, q[2])
    qc.measure(q[2], co)

    return qc


def receiver_accuracy(counts, shots=2000):
    return round(
        100 * sum(
            v for k, v in counts.items()
            if k[0] == '0'
        ) / shots
    )


# BRONZE: Teleportation with corrections
qc1 = teleport(True)
print("Bronze - With corrections:")
print(receiver_accuracy(run(qc1)), "%")


# SILVER: Teleportation without classical corrections
qc2 = teleport(False)
print("Silver - Without corrections:")
print(receiver_accuracy(run(qc2)), "%")


# GOLD: Compare both experiments
print("Gold - Corrected receiver should approach 100%.")
print("Gold - Without corrections, the receiver should approach 50%.")
print("Gold - Entanglement alone cannot send a usable message faster than light.")

Bronze - With corrections:
100 %
Silver - Without corrections:
51 %
Gold - Corrected receiver should approach 100%.
Gold - Without corrections, the receiver should approach 50%.
Gold - Entanglement alone cannot send a usable message faster than light.


Quantum teleportation recovers the intended state when classical measurement results are used for corrections, demonstrating that entanglement alone cannot transmit usable information faster than light.